# Lab 2: Produce a grounded HR answer

## Business problem

Retrieval returns policy text, but an employee needs a direct answer with evidence. The application must identify its sources and refuse to invent missing policy.

## Mission

Connect the Lab 1 index to an LLM while keeping retrieval, context, generation, and citations visible.

## Exercise 1: Rebuild the same index

**Mission:** Reproduce the approved Lab 1 indexing configuration.

**Why it matters:** Changing the source, chunking, metadata, or embedding model creates a different index.

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
EMBEDDING_MODEL = "text-embedding-3-small"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

In [ ]:
sections = []
policy_files = sorted(Path("policies").glob("*.md"))

for policy_file in policy_files:
    parts = policy_file.read_text(encoding="utf-8").split("\n## ")
    for part in parts[1:]:
        heading, text = part.split("\n", 1)
        sections.append({"source": policy_file.name, "section": heading, "text": text.strip()})

print("Policy sections:", len(sections))

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

documents = []
for section in sections:
    pieces = splitter.split_text(section["text"])
    for position, piece in enumerate(pieces):
        metadata = {
            "source": section["source"], "section": section["section"],
            "position": position, "version": "2026.1", "status": "current",
        }
        documents.append(Document(page_content=piece, metadata=metadata))

In [ ]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
vector_store = InMemoryVectorStore.from_documents(documents, embedding=embeddings)
print("Indexed chunks:", len(documents))

## Exercise 2: Retrieve before generating

**Mission:** Make the evidence visible before asking an LLM to answer.

In [ ]:
QUESTION = "Can I expense a $300 train ticket without approval?"
TOP_K = 3

retrieved = vector_store.similarity_search_with_score(QUESTION, k=TOP_K)

for document, score in retrieved:
    print("Similarity:", round(score, 3))
    print(document.metadata["source"], "|", document.metadata["section"])
    print(document.page_content)
    print()

### Retrieval checkpoint

Do not generate until the travel approval section is present. Generation cannot recover evidence that retrieval failed to provide.

## Exercise 3: Build the grounded context

**Mission:** Combine the retrieved text with human-readable citations.

In [ ]:
context_parts = []

for document, score in retrieved:
    citation = document.metadata["source"] + " | " + document.metadata["section"]
    context_parts.append("Source: " + citation + "\n" + document.page_content)

context = "\n\n".join(context_parts)
print(context)

## Exercise 4: Generate the answer

**Mission:** Ask the generation model to answer only from the retrieved policy.

OpenAI creates embeddings and Claude generates the answer. They have separate responsibilities. Replacing either one requires evaluation.

**Industry choices:** This lab uses Anthropic Claude for generation so students see a multi-provider RAG pattern. Common choices in current systems include OpenAI models, Anthropic Claude, Google Gemini, and hosted open-weight models through AWS Bedrock, Azure AI Foundry, or Google Vertex AI. The important LLMOps practice is to record the model name and evaluate a replacement before release.

In [ ]:
from anthropic import Anthropic

client = Anthropic()
GENERATION_MODEL = "claude-haiku-4-5"

system_message = (
    "You are the company HR policy assistant. "
    "Answer only from the supplied context. "
    "If the answer is absent, say: I cannot find that answer in the available HR policy. "
    "End supported answers with the source and section used."
)

user_message = "Context:\n" + context + "\n\nEmployee question:\n" + QUESTION
response = client.messages.create(
    model=GENERATION_MODEL, max_tokens=300, system=system_message,
    messages=[{"role": "user", "content": user_message}],
)
print(response.content[0].text)

## Exercise 5: Test missing evidence

**Mission:** Confirm that the application does not invent a policy.

In [ ]:
MISSING_QUESTION = "Does the company reimburse home internet service?"
missing_docs = vector_store.similarity_search(MISSING_QUESTION, k=TOP_K)

missing_parts = []
for document in missing_docs:
    citation = document.metadata["source"] + " | " + document.metadata["section"]
    missing_parts.append("Source: " + citation + "\n" + document.page_content)

missing_context = "\n\n".join(missing_parts)
missing_message = "Context:\n" + missing_context + "\n\nEmployee question:\n" + MISSING_QUESTION
response = client.messages.create(
    model=GENERATION_MODEL, max_tokens=300, system=system_message,
    messages=[{"role": "user", "content": missing_message}],
)
print(response.content[0].text)

## Exercise 6: Record the request path

**Mission:** Capture enough evidence to troubleshoot one request.

Record the index version, embedding model, generation model, retrieved sources, and elapsed time in production. Avoid logging sensitive employee text unless the data policy permits it.

In [ ]:
request_record = {
    "index_version": "2026.1",
    "embedding_model": EMBEDDING_MODEL,
    "generation_model": GENERATION_MODEL,
    "retrieved_sources": [document.metadata["source"] for document, score in retrieved],
}

print(request_record)

## Exercise 7: Make the request observable

**Mission:** Record the information an operator needs to investigate one answer.

Logs should show the path through the system without storing sensitive employee text by default.

In [ ]:
request_record = {
    "request_id": "demo-001",
    "index_version": "2026.1",
    "embedding_model": EMBEDDING_MODEL,
    "generation_model": GENERATION_MODEL,
    "retrieved_sources": [document.metadata["source"] for document, score in retrieved],
    "retrieved_sections": [document.metadata["section"] for document, score in retrieved],
}

print(request_record)

## Exercise 8: Test an unsupported question

**Mission:** Confirm that the assistant does not pretend that nearby text is an answer.

A production assistant needs an explicit insufficient-evidence response, not only a successful example.

In [ ]:
UNSUPPORTED_QUESTION = "Does the company pay for home internet service?"
unsupported = vector_store.similarity_search(UNSUPPORTED_QUESTION, k=TOP_K)

print("Evidence found:", len(unsupported))
for document in unsupported:
    print(document.metadata["source"], "|", document.metadata["section"])

print("The answer layer must still verify that this evidence answers the question.")

## Exercise 9: Rewrite a vague question

**Mission:** Turn a vague employee question into a search-ready question before retrieval.

In [ ]:
vague_question = "What about salaries?"
rewritten_questions = [
    "What does the 2026 salary review policy say about annual reviews?",
    "What salary bands apply to Cloud and DevOps engineers?",
    "How does an employee request a compensation review?",
]

print("Original:", vague_question)
for rewritten in rewritten_questions:
    print("Search query:", rewritten)

## Exercise 10: Add context to a short chunk

**Mission:** Compare a standalone chunk with a chunk that carries its document context.

In [ ]:
short_chunk = "Section 2 says the limit is 15 days."
contextual_chunk = "HR Policy 2026, Time Off, Section 2: Annual leave. Section 2 says the limit is 15 days."

print("Standalone:", short_chunk)
print("Contextual:", contextual_chunk)

### Lab 2 checkpoint

The request path is now visible: question, retrieve, inspect, optionally rewrite, optionally add context, build context, generate, cite, and record the request. A fluent answer is not proof that retrieval succeeded.